# 07 — E6: Multi-Artifact Counterfactual GLM Debiasing & Confounding Audit (MAC-GLM)

**Novel contribution ($0 extra API cost):**
1. **RQ1 — Confounding Audit on Upstream Benchmark:** Quantifies hidden surface confounding (word-length asymmetry and mid-sentence truncation) between the Markdown and Plain Prose responses in Soumik (2026)'s `STYLE` + `STYLE_BA` dataset.
2. **RQ2 — Judge Bias Fingerprinting:** Fits a regularized Generalized Linear Model (`fit_counterfactual_glm`) to decompose judge preference into Position, Length (`log_word_ratio`), Completeness (`delta_completeness`), and 4 Markdown features (`bold`, `header`, `list`, `struct`).
3. **RQ3 — Zero-Token Post-Hoc Counterfactual Calibration (`do(z = 0)`):** Evaluates out-of-fold counterfactual debiasing (`counterfactual_debias_verdicts`) on cached E3/E5 verdicts vs. prompt/preprocessing interventions (`N1`, `N2`, `S5`).

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
UPSTREAM = PROJECT_ROOT / "data" / "upstream"
RESULTS_E3 = PROJECT_ROOT / "results" / "e3_style_bias"
RESULTS_E5 = PROJECT_ROOT / "results" / "e5_format_normalization"

config = yaml.safe_load((PROJECT_ROOT / "config.yaml").read_text(encoding="utf-8"))

from src.experiments.style_bias import load_style_pairs
from src.experiments.counterfactual_glm import (
    audit_dataset_confounding,
    counterfactual_debias_verdicts,
    evaluate_style_bias_subgroups,
    fit_counterfactual_glm,
    summarize_style_confounding,
)

style_pairs = load_style_pairs(UPSTREAM / "data" / "custom" / "controlled_pairs.jsonl")
print(f"Loaded {len(style_pairs)} STYLE + STYLE_BA pairs.")

## RQ1 — Confounding Audit on Upstream `STYLE` Pairs (No API Calls)

In [ ]:
confound_summary = summarize_style_confounding(style_pairs)
for k, v in confound_summary.items():
    print(f"{k}: {v}")

df_style_art = audit_dataset_confounding(style_pairs)
corr_cols = ["log_word_ratio", "delta_bold", "delta_header", "delta_list", "delta_struct", "delta_completeness"]
print("\nArtifact correlation matrix across 100 STYLE+STYLE_BA pairs:")
df_style_art[corr_cols].corr().round(3)

## Step 1 — Dry-Run Validation of MAC-GLM Pipeline (Zero Cost)

In [ ]:
from src.judges.clients import build_judge

dry_judge = build_judge("dry-run", dry_run=True)
dry_verdicts = [
    dry_judge.judge(p["question"], p["response_a"], p["response_b"]).verdict
    for p in style_pairs
]
dry_fit = fit_counterfactual_glm(style_pairs, dry_verdicts)
dry_cf = counterfactual_debias_verdicts(style_pairs, dry_verdicts, n_folds=5)
print("PASS: MAC-GLM fit and 5-fold counterfactual calibration succeeded in dry-run mode.")
dry_fit["coefficients"]

## RQ2 & RQ3 — Run on Cached Real Judge Verdicts (After Running Notebook 04 / 06)

In [ ]:
for judge_name in config["judges"]["e3_style_bias"]:
    e3_cache_file = RESULTS_E3 / f"{judge_name}_style_full.jsonl"
    if not e3_cache_file.exists():
        print(f"[skip {judge_name}] {e3_cache_file.name} not found yet -- run Notebook 04 first.")
        continue
    rows = [json.loads(line) for line in e3_cache_file.read_text(encoding="utf-8").splitlines() if line.strip()]
    by_pair_rep0 = {r["pair_id"]: r["verdict_raw"] for r in rows if r.get("repeat", 0) == 0}
    paired_items = [p for p in style_pairs if p["id"] in by_pair_rep0]
    verdicts = [by_pair_rep0[p["id"]] for p in paired_items]

    fit = fit_counterfactual_glm(paired_items, verdicts)
    cf = counterfactual_debias_verdicts(paired_items, verdicts, n_folds=5)
    raw_slices = evaluate_style_bias_subgroups(paired_items, verdicts)
    cf_slices = evaluate_style_bias_subgroups(paired_items, cf["debiased_verdicts"])

    print(f"=== Judge: {judge_name} (n={len(paired_items)}) ===")
    print("Raw Style Bias (all / clean / confounded):", raw_slices)
    print("MAC-GLM Debiased Style Bias (all / clean / confounded):", cf_slices)
    display(fit["coefficients"])